# Baseline analytics: AML laundering simulation

This notebook demonstrates how to load exported CSVs from `EventRecorder`, derive simple labels and graph features, and fit lightweight models (LightGBM when available, otherwise scikit-learn fallbacks).

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Adjust to the directory where you exported logs/transactions
EXPORT_DIR = Path(os.getenv('LAUNDER_EXPORT_DIR', 'outputs'))
logs_path = EXPORT_DIR / 'event_logs.csv'
tx_path = EXPORT_DIR / 'transactions.csv'
print('Loading', logs_path, tx_path)
logs = pd.read_csv(logs_path) if logs_path.exists() else pd.DataFrame()
txs = pd.read_csv(tx_path) if tx_path.exists() else pd.DataFrame()
logs.head(), txs.head()

In [ ]:
# Derive labels and simple tabular features
if txs.empty:
    raise SystemExit('No transactions file found. Run the simulator export first.')

feature_cols = ['amount', 'illicit_amount', 'illicit_fraction', 'fee_amount', 'cross_bank', 'cross_currency']
txs['label'] = txs['is_money_laundering'].astype(int)
X = txs[feature_cols].fillna(0)
y = txs['label']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# LightGBM if installed, otherwise GradientBoostingClassifier
try:
    from lightgbm import LGBMClassifier
    model = LGBMClassifier(n_estimators=150, max_depth=-1, random_state=42)
except Exception:
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(random_state=42)

model.fit(X_train, y_train)
from sklearn.metrics import classification_report
preds = model.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
# Graph-derived features (degree/centrality) and minimal GNN-style placeholder
import networkx as nx
G = nx.from_pandas_edgelist(txs, 'sender_account', 'receiver_account', edge_attr='amount', create_using=nx.DiGraph)
centrality = nx.degree_centrality(G)

txs['sender_degree'] = txs['sender_account'].map(centrality).fillna(0)
txs['receiver_degree'] = txs['receiver_account'].map(centrality).fillna(0)

feature_cols = feature_cols + ['sender_degree', 'receiver_degree']
X_graph = txs[feature_cols]
y_graph = txs['label']

# Optional PyTorch Geometric example if installed
try:
    import torch
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv

    node_index = {node: idx for idx, node in enumerate(G.nodes())}
    edge_index = torch.tensor([[node_index[u] for u, v in G.edges()], [node_index[v] for u, v in G.edges()]], dtype=torch.long)
    x = torch.tensor([[centrality.get(node, 0)] for node in G.nodes()], dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)
    conv = SAGEConv(in_channels=1, out_channels=4)
    emb = conv(data.x, data.edge_index)
    print('Graph embeddings shape:', emb.shape)
except Exception as exc:
    print('PyTorch Geometric not installed, skipping GNN demo:', exc)